# 01 — Descarga y benchmark de Qwen3-4B-Instruct-2507 (GGUF Q4_K_M)

**Objetivo:** descargar el modelo cuantizado, montarlo en RAM con `llama-cpp-python` y medir
su desempeño real (RAM ocupada, tokens/s de prompt y de generación) antes de invertir en
ingeniería de prompts.

**Contexto de negocio:** Vivia generará *título + descripción* de propiedades a partir de un
draft estructurado, con un LLM local (decisión D5: sin APIs externas) corriendo en el VPS
(6 cores, 12 GB RAM). Este notebook valida la viabilidad en local.

**Presupuesto esperado** (calculado en la fase de concepción):

| Componente | Estimado |
|---|---|
| Pesos GGUF Q4_K_M | ~2.5 GB |
| KV cache @ 8K contexto (FP16) | ~1.2 GB (≈144 KB/token) |
| Overhead llama.cpp | ~0.3–0.5 GB |
| **Total** | **~3.5–4.2 GB** |

Si la generación local queda por debajo de ~5 tokens/s habría que reconsiderar el tamaño del
modelo (plan B: Llama 3.2 3B o Qwen3-1.7B), porque el VPS será igual o más lento que esta máquina.

## 1. Setup

Las dependencias viven en `notebooks/llm/requirements.txt`. `llama-cpp-python` compila
llama.cpp con CMake al instalarse; si falla, usar los wheels precompilados de CPU
(ver comentario dentro del requirements).

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path

# El GGUF se guarda en models_registry/ (cache local de modelos del proyecto).
# En el VPS este directorio se monta como volumen (MODELS_REGISTRY_HOST), así el
# artefacto queda desde ya en el lugar donde vivirá en producción.
MODELS_DIR = Path("../../models_registry/llm").resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Repo público de unsloth (el repo oficial de Qwen no expone GGUF descargable).
REPO_ID = "unsloth/Qwen3-4B-Instruct-2507-GGUF"
FILENAME = "Qwen3-4B-Instruct-2507-Q4_K_M.gguf"  # ~2.5 GB

# Parámetros de carga. En el VPS (6 cores) usar n_threads=5 para dejar aire al resto.
N_CTX = 8192
N_THREADS = 3

print(f"Los modelos se guardarán en: {MODELS_DIR}")

Los modelos se guardarán en: /home/aleosh/Documentos/Ingeniería en Software/9no Cuatrimestre/Integrador/vps/vivia-ai/models_registry/llm


## 2. Descarga del modelo

`hf_hub_download` es idempotente: si el archivo ya existe y está completo, no re-descarga.
El repo es público, no requiere token de Hugging Face.

In [8]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    local_dir=MODELS_DIR,
)

size_gb = Path(model_path).stat().st_size / 1e9
print(f"Modelo en: {model_path}")
print(f"Tamaño en disco: {size_gb:.2f} GB")

Modelo en: /home/aleosh/Documentos/Ingeniería en Software/9no Cuatrimestre/Integrador/vps/vivia-ai/models_registry/llm/Qwen3-4B-Instruct-2507-Q4_K_M.gguf
Tamaño en disco: 2.50 GB


## 3. Carga en RAM

Se mide la RAM del proceso antes y después de montar el modelo para validar el presupuesto.
El GGUF trae su propio *chat template*, así que no hace falta especificar `chat_format`.

In [9]:
import time

import psutil
from llama_cpp import Llama

proceso = psutil.Process()
ram_antes = proceso.memory_info().rss / 1e9

t0 = time.perf_counter()
llm = Llama(
    model_path=model_path,
    n_ctx=N_CTX,
    n_threads=N_THREADS,
    verbose=False,
)
t_carga = time.perf_counter() - t0

ram_despues = proceso.memory_info().rss / 1e9
print(f"Tiempo de carga: {t_carga:.1f} s")
print(f"RAM del proceso: {ram_antes:.2f} GB → {ram_despues:.2f} GB (Δ {ram_despues - ram_antes:.2f} GB)")
print(f"Contexto configurado: {N_CTX} tokens | Threads: {N_THREADS}")

Tiempo de carga: 1.6 s
RAM del proceso: 5.72 GB → 5.74 GB (Δ 0.03 GB)
Contexto configurado: 8192 tokens | Threads: 3


## 4. Smoke test

Una generación corta en español para confirmar que el modelo responde coherente antes de medir nada.

In [10]:
respuesta = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "En una sola oración: ¿qué hace atractiva a una casa con jardín en México?"},
    ],
    max_tokens=80,
    temperature=0.7,
)
print(respuesta["choices"][0]["message"]["content"])

Una casa con jardín en México es atractiva porque combina la belleza natural del entorno con un estilo de vida tranquilo y conectado con la tierra, reflejando la tradición y el equilibrio entre el hogar y la naturaleza.


## 5. Benchmark

Se hace *streaming* de la respuesta para separar las dos métricas que importan:

- **Prompt processing** (leer el draft): tiempo hasta el primer token.
- **Generación** (escribir la descripción): tokens/s sostenidos después del primer token.

Se corre 3 veces con prompts distintos para evitar efectos de caché de prefijo.

In [11]:
def benchmark_generacion(prompt: str, max_tokens: int = 200) -> dict:
    """Mide prompt processing (t hasta 1er token) y velocidad de generación."""
    n_tokens_prompt = len(llm.tokenize(prompt.encode("utf-8")))
    t0 = time.perf_counter()
    t_primer_token = None
    n_generados = 0
    for chunk in llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.7,
        stream=True,
    ):
        delta = chunk["choices"][0]["delta"]
        if delta.get("content"):
            if t_primer_token is None:
                t_primer_token = time.perf_counter()
            n_generados += 1
    t_fin = time.perf_counter()

    t_prompt = t_primer_token - t0
    t_gen = t_fin - t_primer_token
    return {
        "tokens_prompt": n_tokens_prompt,
        "tokens_generados": n_generados,
        "prompt_tok_s": n_tokens_prompt / t_prompt if t_prompt > 0 else float("nan"),
        "generacion_tok_s": (n_generados - 1) / t_gen if t_gen > 0 else float("nan"),
        "tiempo_total_s": t_fin - t0,
    }


PROMPTS_BENCH = [
    "Describe en unas 150 palabras las ventajas de vivir en un departamento céntrico con dos recámaras, "
    "dos baños completos, cocina integral, un cajón de estacionamiento y amenidades como gimnasio y roof garden.",
    "Redacta en unas 150 palabras un texto atractivo sobre una casa de dos plantas con jardín amplio, "
    "tres recámaras, dos baños y medio, cochera techada para dos autos y cuarto de servicio.",
    "Escribe en unas 150 palabras una reseña comercial de un loft moderno de 80 metros cuadrados con "
    "acabados de lujo, doble altura, terraza privada y seguridad las 24 horas.",
]

resultados = [benchmark_generacion(p) for p in PROMPTS_BENCH]

import pandas as pd

df_bench = pd.DataFrame(resultados)
display(df_bench.round(1))

gen_tok_s = df_bench["generacion_tok_s"].mean()
prompt_tok_s = df_bench["prompt_tok_s"].mean()
print(f"\nPromedios — prompt: {prompt_tok_s:.1f} tok/s | generación: {gen_tok_s:.1f} tok/s")
print(f"Estimado para una salida de ~300 tokens (título + descripción): {300 / gen_tok_s:.0f} s")

,tokens_prompt,tokens_generados,prompt_tok_s,generacion_tok_s,tiempo_total_s
0,56,200,26.2,8.3,26.0
1,54,200,30.8,8.3,25.6
2,50,200,31.1,8.4,25.2



Promedios — prompt: 29.4 tok/s | generación: 8.4 tok/s
Estimado para una salida de ~300 tokens (título + descripción): 36 s


## 6. Conclusiones

> Llenar después de ejecutar el benchmark en cada máquina.

| Métrica | Local (8c / 19 GB) | VPS (6c / 12 GB) |
|---|---|---|
| RAM ocupada por el modelo | | |
| Prompt processing (tok/s) | | |
| Generación (tok/s) | | |
| Tiempo por anuncio (~300 tok) | | |

**Criterio de decisión:** con ≥5 tok/s de generación en el VPS, el flujo asíncrono
(RabbitMQ → worker con concurrencia 1) entrega un anuncio en ≤60 s, aceptable para el negocio.
Por debajo de eso, evaluar Llama 3.2 3B (Q4, ~1.9 GB) o Qwen3-1.7B.

**Siguiente paso:** `02_generacion_titulo_descripcion.ipynb` — prompts y salida estructurada
sobre el testbench de propiedades.